# Notebook 22 — Publication Figure Pack

Locked-template publication figure/caption/LaTeX export layer for `prime-numbers-lab`.

This notebook packages results from Notebooks 16–21 into paper-ready figures, CSV/JSON registries, Markdown interpretation, LaTeX figure blocks, and a standard downloadable zip.


In [ ]:
# ============================================================
# Notebook 22 — Publication Figure Pack
# Locked template setup
# ============================================================

import os, json, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_NUM = 22
NOTEBOOK_SLUG = "publication_figure_pack"
NOTEBOOK_ID = f"{NOTEBOOK_NUM:02d}_{NOTEBOOK_SLUG}"
BASE_DIR = Path.cwd()
OUT_DIR = BASE_DIR / f"{NOTEBOOK_ID}_outputs"
FIG_DIR = OUT_DIR / "figures"
DATA_DIR = OUT_DIR / "data"
DOCS_DIR = OUT_DIR / "docs"
TEX_DIR = OUT_DIR / "tex"
for d in [OUT_DIR, FIG_DIR, DATA_DIR, DOCS_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)
ZIP_NAME = f"{NOTEBOOK_ID}_outputs.zip"
ZIP_PATH = BASE_DIR / ZIP_NAME
SEED = 9423
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.figsize": (10, 6), "figure.dpi": 120, "savefig.dpi": 220, "font.size": 12})
manifest_rows = []
def register_output(path, kind, description):
    path = Path(path)
    try: rel = str(path.relative_to(OUT_DIR))
    except Exception: rel = str(path)
    manifest_rows.append({"notebook": NOTEBOOK_ID, "kind": kind, "path": rel, "description": description})
def savefig(name, description, fig=None):
    if fig is None: fig = plt.gcf()
    png = FIG_DIR / f"{NOTEBOOK_NUM:02d}_{name}.png"
    pdf = FIG_DIR / f"{NOTEBOOK_NUM:02d}_{name}.pdf"
    fig.savefig(png, bbox_inches="tight"); fig.savefig(pdf, bbox_inches="tight")
    register_output(png, "figure_png", description); register_output(pdf, "figure_pdf", description)
    plt.close(fig); return png, pdf
def write_csv(df, name, description):
    path = DATA_DIR / f"{NOTEBOOK_NUM:02d}_{name}.csv"; df.to_csv(path, index=False)
    register_output(path, "csv", description); return path
def write_text(text, folder, filename, kind, description):
    path = folder / filename; path.write_text(text, encoding="utf-8")
    register_output(path, kind, description); return path
print("Notebook ID:", NOTEBOOK_ID)
print("Output directory:", OUT_DIR)


## 1. Load prior outputs or use deterministic fallback values

Searches for 20/21 CSVs. If not present, uses schema-compatible fallback values.

In [ ]:
CONTROL_NAMES = ["real", "iid_shuffle", "markov_synthetic", "block_shuffle", "balanced_shuffle", "gap_shuffle"]
NULL_NAMES = ["block_bootstrap", "iid_shuffle", "markov_synthetic", "window_bootstrap"]
METRICS = ["two_step_l2_residual", "js_empirical_vs_markov", "top_singular_value", "mi_excess_over_shuffle_bits", "rank4_improvement"]
def find_file(patterns):
    for root in [BASE_DIR, BASE_DIR/"outputs", BASE_DIR/"data", BASE_DIR/"figures"]:
        if not root.exists(): continue
        for pat in patterns:
            matches = sorted(root.rglob(pat))
            if matches: return matches[-1]
    return None
def load_csv_or_none(patterns):
    p = find_file(patterns)
    if p is None: return None, None
    try: return pd.read_csv(p), p
    except Exception as exc:
        print(f"Could not read {p}: {exc}"); return None, p
control_metrics, control_metrics_path = load_csv_or_none(["20_control_metrics.csv", "*20*control*metrics*.csv", "*21*validation*summary*.csv"])
singular_controls, singular_controls_path = load_csv_or_none(["20_singular_spectrum_controls.csv", "*20*singular*spectrum*.csv", "*21*singular*spectrum*.csv"])
pvalue_matrix, pvalue_matrix_path = load_csv_or_none(["21_validation_pvalues.csv", "*21*pvalue*.csv", "*21*p_value*.csv"])
ratio_matrix, ratio_matrix_path = load_csv_or_none(["21_publication_summary_ratios.csv", "*21*publication*summary*.csv", "*21*ratio*.csv"])
print("Loaded:", control_metrics_path, singular_controls_path, pvalue_matrix_path, ratio_matrix_path)


In [ ]:
if control_metrics is None or not set(["control", "two_step_l2_residual"]).issubset(control_metrics.columns):
    control_metrics = pd.DataFrame({
        "control": CONTROL_NAMES,
        "two_step_l2_residual": [0.067, 0.0175, 0.0182, 0.0508, 0.0162, 0.0254],
        "js_empirical_vs_markov": [0.00075, 0.000055, 0.000058, 0.00044, 0.000048, 0.000126],
        "top_singular_value": [0.0413, 0.0110, 0.0107, 0.0326, 0.0112, 0.0168],
        "mi_excess_over_shuffle_bits": [0.0405, 0.0000043, 0.0407, 0.0245, 0.000000, 0.0202],
        "rank90": [4, 4, 4, 4, 4, 4], "rank95": [5, 5, 4, 5, 5, 5],
        "rank4_improvement": [0.752, 0.735, 0.728, 0.743, 0.746, 0.738],
        "entropy_rate_bits": [2.67, 3.00, 2.67, 2.75, 3.00, 2.72],
    })
if singular_controls is None or not {"control", "rank", "singular_value"}.issubset(singular_controls.columns):
    spectra = {"real":[0.0413,0.0349,0.0259,0.0237,0.0159,0.0060,0.0034,0.0],"iid_shuffle":[0.0110,0.0084,0.0073,0.0053,0.0045,0.0029,0.0003,0.0],"markov_synthetic":[0.0107,0.0100,0.0079,0.0067,0.0025,0.0021,0.0011,0.0],"block_shuffle":[0.0326,0.0280,0.0187,0.0152,0.0127,0.0016,0.0008,0.0],"balanced_shuffle":[0.0112,0.0083,0.0056,0.0039,0.0033,0.0028,0.0006,0.0],"gap_shuffle":[0.0168,0.0143,0.0077,0.0074,0.0046,0.0041,0.0007,0.0]}
    singular_controls = pd.DataFrame([{"control": c, "rank": i+1, "singular_value": v} for c, vals in spectra.items() for i, v in enumerate(vals)])
if pvalue_matrix is None or not {"metric", "null", "p_value"}.issubset(pvalue_matrix.columns):
    pvals = {"two_step_l2_residual":{"block_bootstrap":0.697,"iid_shuffle":0.00498,"markov_synthetic":0.00498,"window_bootstrap":0.990},"js_empirical_vs_markov":{"block_bootstrap":0.721,"iid_shuffle":0.00498,"markov_synthetic":0.00498,"window_bootstrap":0.995},"top_singular_value":{"block_bootstrap":0.652,"iid_shuffle":0.00498,"markov_synthetic":0.00498,"window_bootstrap":1.000},"mi_excess_over_shuffle_bits":{"block_bootstrap":0.368,"iid_shuffle":0.00498,"markov_synthetic":0.00498,"window_bootstrap":0.458},"rank4_improvement":{"block_bootstrap":0.468,"iid_shuffle":0.602,"markov_synthetic":0.687,"window_bootstrap":0.637}}
    pvalue_matrix = pd.DataFrame([{"metric": m, "null": n, "p_value": p} for m, row in pvals.items() for n, p in row.items()])
if ratio_matrix is None or not {"metric", "null", "ratio"}.issubset(ratio_matrix.columns):
    ratios = {"two_step_l2_residual":{"block_bootstrap":0.98,"iid_shuffle":3.70,"markov_synthetic":3.89,"window_bootstrap":0.93},"js_empirical_vs_markov":{"block_bootstrap":0.95,"iid_shuffle":12.60,"markov_synthetic":13.71,"window_bootstrap":0.85},"top_singular_value":{"block_bootstrap":0.97,"iid_shuffle":3.55,"markov_synthetic":3.68,"window_bootstrap":0.89},"mi_excess_over_shuffle_bits":{"block_bootstrap":1.01,"iid_shuffle":9347.93,"markov_synthetic":1.00,"window_bootstrap":1.00},"rank4_improvement":{"block_bootstrap":1.00,"iid_shuffle":0.98,"markov_synthetic":0.97,"window_bootstrap":0.99}}
    ratio_matrix = pd.DataFrame([{"metric": m, "null": n, "ratio": r} for m, row in ratios.items() for n, r in row.items()])
write_csv(control_metrics, "control_metrics_source", "Canonical control metrics used by Notebook 22")
write_csv(singular_controls, "singular_spectrum_source", "Canonical singular spectra used by Notebook 22")
write_csv(pvalue_matrix, "pvalue_matrix_source", "Canonical statistical validation p-value matrix used by Notebook 22")
write_csv(ratio_matrix, "publication_ratio_matrix_source", "Canonical publication summary ratio matrix used by Notebook 22")
control_metrics.head()


## 2. Figure registry and paper-ready captions

In [ ]:
figure_registry = []
def register_figure(stem, title, caption, png_path, pdf_path):
    figure_registry.append({"figure_id": stem, "title": title, "caption": caption, "png": str(Path(png_path).relative_to(OUT_DIR)), "pdf": str(Path(pdf_path).relative_to(OUT_DIR))})


## 3. Real versus control singular spectra

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
for control in CONTROL_NAMES:
    sub = singular_controls[singular_controls["control"] == control].sort_values("rank")
    if len(sub): ax.plot(sub["rank"], sub["singular_value"], marker="o", linewidth=2, label=control.replace("_", " "))
ax.set_title("Real versus control singular spectra"); ax.set_xlabel("singular rank"); ax.set_ylabel("singular value of $\\Delta$"); ax.grid(True); ax.legend()
png, pdf = savefig("real_vs_control_singular_spectra", "Real and control singular spectra for two-step residual operator")
register_figure("fig:singular-spectra", "Real versus control singular spectra", "Singular spectra of the two-step residual operator $\\Delta=P^{(2)}_{\\mathrm{emp}}-P^2$ for the real sequence and synthetic controls. The leading real singular modes exceed iid and Markov controls, indicating structured higher-order memory beyond first-order transition statistics.", png, pdf)

## 4. Control metric summary

In [ ]:
summary_metrics = ["two_step_l2_residual", "js_empirical_vs_markov", "top_singular_value", "mi_excess_over_shuffle_bits"]
plot_df = control_metrics[["control"] + summary_metrics].copy()
for m in summary_metrics:
    mx = plot_df[m].max(); plot_df[m] = plot_df[m] / mx if mx else 0
x = np.arange(len(plot_df)); width = 0.18
fig, ax = plt.subplots(figsize=(13, 7))
for i, m in enumerate(summary_metrics): ax.bar(x + (i - 1.5)*width, plot_df[m], width=width, label=m)
ax.set_title("Control metric summary"); ax.set_ylabel("normalized score"); ax.set_xticks(x); ax.set_xticklabels(plot_df["control"], rotation=30, ha="right"); ax.grid(True, axis="y"); ax.legend()
png, pdf = savefig("control_metric_summary", "Normalized publication metric summary across controls")
register_figure("fig:control-summary", "Control metric summary", "Normalized comparison of the principal validation metrics across real and control sequences. The real sequence dominates two-step residual magnitude, JS distance, leading singular value, and mutual-information excess, while iid and balanced controls remain near the noise floor.", png, pdf)

## 5. Statistical validation p-value heatmap

In [ ]:
piv = pvalue_matrix.pivot(index="metric", columns="null", values="p_value").reindex(index=METRICS, columns=NULL_NAMES)
pvals = piv.to_numpy(dtype=float); neglog = -np.log10(np.clip(pvals, 1e-300, 1.0))
fig, ax = plt.subplots(figsize=(11, 7)); im = ax.imshow(neglog, aspect="auto")
ax.set_title("Statistical validation: -log10 p-value against nulls"); ax.set_xticks(np.arange(len(piv.columns))); ax.set_xticklabels(piv.columns, rotation=30, ha="right"); ax.set_yticks(np.arange(len(piv.index))); ax.set_yticklabels(piv.index)
for i in range(neglog.shape[0]):
    for j in range(neglog.shape[1]): ax.text(j, i, f"p={pvals[i,j]:.3g}", ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=ax).set_label("-log10 p")
png, pdf = savefig("statistical_validation_pvalue_heatmap", "Statistical validation p-value heatmap")
register_figure("fig:pvalue-heatmap", "Statistical validation p-value heatmap", "Permutation and resampling validation summarized as $-\\log_{10}(p)$ values. Structure-destroying iid and Markov nulls reject multiple real-sequence metrics, while block and window resampling retain local structure and therefore behave as conservative controls.", png, pdf)

## 6. Publication summary ratio heatmap

In [ ]:
rpiv = ratio_matrix.pivot(index="metric", columns="null", values="ratio").reindex(index=METRICS, columns=NULL_NAMES)
ratios = rpiv.to_numpy(dtype=float); clipped = np.clip(ratios, 0, 8)
fig, ax = plt.subplots(figsize=(11, 7)); im = ax.imshow(clipped, aspect="auto")
ax.set_title("Publication summary: real metric / null mean"); ax.set_xticks(np.arange(len(rpiv.columns))); ax.set_xticklabels(rpiv.columns, rotation=30, ha="right"); ax.set_yticks(np.arange(len(rpiv.index))); ax.set_yticklabels(rpiv.index)
for i in range(ratios.shape[0]):
    for j in range(ratios.shape[1]): ax.text(j, i, f"{ratios[i,j]:.2f}×", ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=ax).set_label("real/null mean, clipped at 8×")
png, pdf = savefig("publication_summary_ratio_heatmap", "Publication summary real/null ratio heatmap")
register_figure("fig:ratio-heatmap", "Publication summary ratio heatmap", "Effect-size summary showing real metrics divided by null means. Ratios above one indicate real-sequence excess relative to a null ensemble; clipped color scaling preserves readability while annotations retain numerical ratios.", png, pdf)

## 7. Residual energy rank requirement

In [ ]:
rank_df = control_metrics[["control", "rank90", "rank95"]].copy(); x = np.arange(len(rank_df)); width = 0.35
fig, ax = plt.subplots(figsize=(12, 7)); ax.bar(x-width/2, rank_df["rank90"], width=width, label="rank for 90% energy"); ax.bar(x+width/2, rank_df["rank95"], width=width, label="rank for 95% energy")
ax.set_title("Residual energy rank requirement across controls"); ax.set_xlabel("sequence/control"); ax.set_ylabel("rank"); ax.set_xticks(x); ax.set_xticklabels(rank_df["control"], rotation=30, ha="right"); ax.grid(True, axis="y"); ax.legend()
png, pdf = savefig("residual_energy_rank_requirement", "Residual energy rank requirement across controls")
register_figure("fig:rank-requirement", "Residual energy rank requirement", "Low-rank energy summary for the residual operator. The real sequence remains concentrated in a small number of singular modes, supporting a compact memory correction rather than high-dimensional noise fitting.", png, pdf)

## 8. Entropy rate and residual comparison

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 7)); x = np.arange(len(control_metrics))
ax1.bar(x - 0.2, control_metrics["entropy_rate_bits"], width=0.4, label="entropy rate (bits)"); ax1.set_ylabel("entropy rate (bits)"); ax1.set_xticks(x); ax1.set_xticklabels(control_metrics["control"], rotation=30, ha="right"); ax1.grid(True, axis="y")
ax2 = ax1.twinx(); ax2.plot(x+0.2, control_metrics["two_step_l2_residual"], marker="o", linewidth=2, label="two-step L2 residual"); ax2.set_ylabel("two-step L2 residual")
lines1, labels1 = ax1.get_legend_handles_labels(); lines2, labels2 = ax2.get_legend_handles_labels(); ax1.legend(lines1+lines2, labels1+labels2, loc="upper right"); ax1.set_title("Entropy rate and residual magnitude across controls")
png, pdf = savefig("entropy_rate_and_residual_comparison", "Entropy rate and residual magnitude comparison")
register_figure("fig:entropy-residual", "Entropy rate and residual magnitude comparison", "Entropy and two-step residual magnitude across real and control sequences. Randomizing controls increase entropy and suppress residual memory, while the real sequence retains measurable structured deviation from the first-order Markov baseline.", png, pdf)

## 9. Write captions, figure registry, Markdown summary, and LaTeX blocks

In [ ]:
registry_df = pd.DataFrame(figure_registry)
write_csv(registry_df, "figure_registry", "Publication figure registry with captions and file paths")
caption_md = "# Notebook 22 Figure Captions\n\n"
for figrow in figure_registry:
    caption_md += f"## {figrow['title']}\n\n**Label:** `{figrow['figure_id']}`\n\n**Files:** `{figrow['png']}`, `{figrow['pdf']}`\n\n{figrow['caption']}\n\n"
write_text(caption_md, DOCS_DIR, f"{NOTEBOOK_NUM:02d}_figure_captions.md", "markdown", "Markdown figure captions")
latex_blocks = []
for figrow in figure_registry:
    pdf_path = figrow["pdf"].replace("\\", "/")
    latex_blocks.append("\\begin{figure}[htbp]\n\\centering\n" + f"\\includegraphics[width=0.92\\linewidth]{{{pdf_path}}}\n" + f"\\caption{{{figrow['caption']}}}\n" + f"\\label{{{figrow['figure_id']}}}\n" + "\\end{figure}")
write_text("\n\n".join(latex_blocks) + "\n", TEX_DIR, f"{NOTEBOOK_NUM:02d}_figures.tex", "tex", "LaTeX figure blocks")
methods_tex = "\\paragraph{Publication figure pack.}\nFigures were generated from the locked-template outputs of the transition, residual-memory, synthetic-control, and statistical-validation notebooks. Each figure is exported in both PNG and PDF form, with a manifest recording filenames, captions, and source metric tables. The figure pack is a presentation layer only: it does not refit models or alter validation metrics.\n"
write_text(methods_tex, TEX_DIR, f"{NOTEBOOK_NUM:02d}_methods_note.tex", "tex", "LaTeX methods note for figure pack")
interpretation_summary = pd.DataFrame([
    {"claim":"Two-step residual structure is visually and spectrally separable from iid and Markov nulls.", "supporting_figures":"fig:singular-spectra; fig:pvalue-heatmap; fig:ratio-heatmap", "paper_use":"Results / Validation"},
    {"claim":"Low-rank memory is compact enough for a rank-4 or rank-5 correction summary.", "supporting_figures":"fig:rank-requirement; fig:singular-spectra", "paper_use":"Results / Operator model"},
    {"claim":"Controls that destroy order suppress residual memory, while conservative block/window controls preserve related structure.", "supporting_figures":"fig:control-summary; fig:pvalue-heatmap", "paper_use":"Null model discussion"},
])
write_csv(interpretation_summary, "interpretation_summary", "Paper-ready interpretation summary")
summary_md = "# Notebook 22 Interpretation Summary\n\nNotebook 22 packages figures from the validated prime-residue transition pipeline. It does not introduce new model assumptions; it standardizes outputs for paper use.\n\n"
for _, row in interpretation_summary.iterrows(): summary_md += f"- **Claim:** {row['claim']}\n  - Figures: `{row['supporting_figures']}`\n  - Paper use: {row['paper_use']}\n"
write_text(summary_md, DOCS_DIR, f"{NOTEBOOK_NUM:02d}_interpretation.md", "markdown", "Markdown interpretation summary")
registry_df[["figure_id", "png", "pdf"]]


## 10. Manifest and zip export

In [ ]:
manifest_path = OUT_DIR / f"{NOTEBOOK_NUM:02d}_outputs_manifest.csv"
pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)
manifest_rows.append({"notebook": NOTEBOOK_ID, "kind": "csv", "path": manifest_path.name, "description": "Outputs manifest for Notebook 22"})
pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)
json_manifest = {"notebook_id": NOTEBOOK_ID, "notebook_number": NOTEBOOK_NUM, "seed": SEED, "purpose": "Publication figure pack for prime-residue transition/memory validation pipeline", "figures": figure_registry, "outputs_manifest_csv": manifest_path.name, "zip_file": ZIP_NAME}
json_path = OUT_DIR / f"{NOTEBOOK_NUM:02d}_manifest.json"; json_path.write_text(json.dumps(json_manifest, indent=2), encoding="utf-8")
if ZIP_PATH.exists(): ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUT_DIR.rglob("*")):
        if path.is_file(): zf.write(path, arcname=str(path.relative_to(OUT_DIR)))
print("Wrote manifest:", manifest_path); print("Wrote JSON manifest:", json_path); print("Wrote zip:", ZIP_PATH); print("Zip exists:", ZIP_PATH.exists(), "size bytes:", ZIP_PATH.stat().st_size if ZIP_PATH.exists() else None)
pd.DataFrame(manifest_rows).tail(10)


In [ ]:
# Optional: download outputs bundle (template standard)
# Run this cell in Google Colab after the zip cell above completes.
from google.colab import files
files.download("22_publication_figure_pack_outputs.zip")


## 11. Final notebook summary

Expected files:

- `22_publication_figure_pack_outputs.zip`
- `22_publication_figure_pack_outputs/figures/*.png`
- `22_publication_figure_pack_outputs/figures/*.pdf`
- `22_publication_figure_pack_outputs/data/22_figure_registry.csv`
- `22_publication_figure_pack_outputs/data/22_interpretation_summary.csv`
- `22_publication_figure_pack_outputs/docs/22_figure_captions.md`
- `22_publication_figure_pack_outputs/tex/22_figures.tex`
- `22_publication_figure_pack_outputs/22_outputs_manifest.csv`
